In [1]:
# INSTALLING LIBRARIES
!pip install -q transformers peft bitsandbytes accelerate datasets
!pip install trl==0.9.6

In [2]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer

In [3]:
# Loading Dataset from File
dataset = load_dataset("json", data_files="dataset.json", split="train")
print(f"Dataset loaded. Number of examples: {len(dataset)}")
print(f"First example: \n{dataset[0]['text']}")

Dataset loaded. Number of examples: 223
First example: 
<s>[INST] Data: [Journal: 'Woke up late, skipped breakfast, feeling guilty but oddly focused later.']
Task: Write an insight. [/INST] You had a rocky start but still found focus later—small routine slips don't erase the productive momentum you built. </s>


In [4]:
# Model and Tokenizer Setup
model_id = "microsoft/Phi-3-mini-4k-instruct"

# bnb config for 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Loading model and tokenizer
print("Loading model and tokenizer...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading model and tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
# LoRA Config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Phi-3's target modules
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_up_proj", "down_proj"],
)

# Adding LoRA to the model
model = get_peft_model(model, lora_config)

In [6]:
import trl
print(trl.__version__)


0.9.6


In [7]:
# Training Arguments
training_args = TrainingArguments(
    output_dir="phi3-insight-writer",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=3,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
)

def formatting_func(example):
    return example["text"]

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=1024,
    formatting_func=formatting_func,
    packing=False,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/223 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:413: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


In [8]:
# Training
trainer.train()
print("Training finished!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dhinwaji (dhinwaji-iit-bombay) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,2.480600
20,1.425100
30,1.190600
40,1.033100
50,1.022800
60,0.935900
70,0.827700
80,0.808600


Training finished!


In [9]:
# Saveing the Fine-Tuned Adapters
adapter_model_name = "insight_writer_adapters"
trainer.model.save_pretrained(adapter_model_name)
print(f"Adapters saved to {adapter_model_name}")

# Zipping Adapters for Download
!zip -r insight_writer_adapters.zip insight_writer_adapters
print("Adapters zipped")

Adapters saved to insight_writer_adapters
  adding: insight_writer_adapters/ (stored 0%)
  adding: insight_writer_adapters/adapter_config.json (deflated 57%)
  adding: insight_writer_adapters/adapter_model.safetensors (deflated 7%)
  adding: insight_writer_adapters/README.md (deflated 65%)
Adapters zipped
